# Step 2: Build Sensor Features (with Horizon Mapping)
## Modified to create column_step_map.json for horizon-based filtering

This notebook builds the sensor feature matrix and creates a mapping of each feature column to its process SeqNo.

**NEW: Includes caching for expensive parquet processing steps!**

In [1]:
import time
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import gc
import pyarrow.parquet as pq
import json

start_time = time.time()
print("=" * 60)
print("Step 2: Building Sensor Features (with Horizon Mapping)")
print("=" * 60)

Step 2: Building Sensor Features (with Horizon Mapping)


In [2]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"✓ Project root: {project_root}")
print(f"✓ Current directory: {Path.cwd()}")
print(f"✓ Data directory exists: {(Path.cwd() / 'data').exists()}")
print(f"✓ Outputs directory exists: {(Path.cwd() / 'outputs').exists()}")

✓ Project root: d:\capstone_pipeline
✓ Current directory: d:\capstone_pipeline
✓ Data directory exists: True
✓ Outputs directory exists: True


In [3]:
# Setup paths
data_dir = Path("data")
output_dir = Path("outputs")
features_dir = output_dir / "features"
cache_dir = output_dir / "cache"  # NEW: cache directory
features_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

sentinel = features_dir / "02_sensor.done"
force = False  # Set to True to force rebuild
use_cache = True  # Set to False to disable cache (forces reprocessing)

if sentinel.exists() and not force:
    print(f"[OK] Sensor features already built (found {sentinel})")
    print(f"  Set force=True to rebuild")
else:
    print("Building sensor features...")

[OK] Sensor features already built (found outputs\features\02_sensor.done)
  Set force=True to rebuild


In [4]:
# Check input files
mean_file = data_dir / "fd_mean.parquet"
stdev_file = data_dir / "fd_stddev.parquet"

if not mean_file.exists():
    raise FileNotFoundError(
        f"Could not find {mean_file}\n"
        f"Please ensure fd_mean.parquet is in the data/ directory."
    )

if not stdev_file.exists():
    raise FileNotFoundError(
        f"Could not find {stdev_file}\n"
        f"Please ensure fd_stddev.parquet is in the data/ directory."
    )

In [5]:
def process_parquet_batches(file_path, suffix, wafer_list):
    """Process parquet file in batches"""
    print(f"  Reading {file_path.name}...")
    
    parquet_file = pq.ParquetFile(file_path)
    result = {wafer: {} for wafer in wafer_list}
    
    print(f"  Processing {parquet_file.metadata.num_row_groups} row groups...")
    value_col = 'MeanValue' if suffix == 'MEAN' else 'StDevValue'
    
    for batch in tqdm(parquet_file.iter_batches(batch_size=1000000), 
                     total=parquet_file.metadata.num_row_groups):
        df = batch.to_pandas()
        df['feature_name'] = df['SENSOR'] + "__" + df['STEP'] + "__" + suffix
        grouped = df.groupby(['WAFER_SCRIBE', 'feature_name'])[value_col].mean()
        
        for (wafer, feature), value in grouped.items():
            if wafer in result:
                result[wafer][feature] = value
        
        del df, grouped
        gc.collect()
    
    print(f"  Converting to DataFrame...")
    result_df = pd.DataFrame.from_dict(result, orient='index')
    result_df.index.name = 'WAFER_SCRIBE'
    
    return result_df

In [6]:
# Get list of unique wafers from target
target_file = features_dir / "target.parquet"
if target_file.exists():
    target_df = pd.read_parquet(target_file)
    wafer_list = target_df['WAFER_SCRIBE'].unique().tolist()
    print(f"\nUsing {len(wafer_list)} wafers from target file")
else:
    wafer_list = None
    print("\nWARNING: No target file found, will discover wafers from data")


Using 16817 wafers from target file


## Process Mean Features (with Caching)

This cell checks for a cached version first. If found, loads from cache. Otherwise processes and saves to cache.

In [7]:
# Process mean features with caching
mean_cache_file = cache_dir / "mean_pivot.parquet"

if use_cache and mean_cache_file.exists() and not force:
    print("\n📦 Loading mean features from cache...")
    mean_pivot = pd.read_parquet(mean_cache_file)
    print(f"  ✓ Loaded from cache: {mean_pivot.shape}")
    print(f"  ✓ Memory: {mean_pivot.memory_usage(deep=True).sum() / 1e6:.2f} MB")
else:
    print("\n⚙️  Processing mean features from scratch...")
    if use_cache and mean_cache_file.exists():
        print("  (cache exists but force=True, reprocessing)")
    
    if wafer_list:
        mean_pivot = process_parquet_batches(mean_file, 'MEAN', wafer_list)
    else:
        df_mean = pd.read_parquet(mean_file)
        df_mean['feature_name'] = df_mean['SENSOR'] + "__" + df_mean['STEP'] + "__MEAN"
        mean_pivot = df_mean.pivot_table(
            index='WAFER_SCRIBE',
            columns='feature_name',
            values='MeanValue',
            aggfunc='mean'
        )
        del df_mean
        gc.collect()
    
    print(f"  Mean features shape: {mean_pivot.shape}")
    print(f"  Memory: {mean_pivot.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    
    # Save to cache
    print(f"\n  💾 Saving to cache: {mean_cache_file}")
    mean_pivot.to_parquet(mean_cache_file)
    print(f"  ✓ Cache saved ({mean_cache_file.stat().st_size / 1e6:.2f} MB)")


⚙️  Processing mean features from scratch...
  Reading fd_mean.parquet...
  Processing 1390 row groups...


 12%|█▏        | 172/1390 [11:19<1:20:13,  3.95s/it]


  Converting to DataFrame...
  Mean features shape: (16817, 6534)
  Memory: 879.39 MB

  💾 Saving to cache: outputs\cache\mean_pivot.parquet
  ✓ Cache saved (486.33 MB)


## Process Stdev Features (with Caching)

This cell checks for a cached version first. If found, loads from cache. Otherwise processes and saves to cache.

In [8]:
# Process stdev features with caching
stdev_cache_file = cache_dir / "stdev_pivot.parquet"

if use_cache and stdev_cache_file.exists() and not force:
    print("\n📦 Loading stdev features from cache...")
    stdev_pivot = pd.read_parquet(stdev_cache_file)
    print(f"  ✓ Loaded from cache: {stdev_pivot.shape}")
    print(f"  ✓ Memory: {stdev_pivot.memory_usage(deep=True).sum() / 1e6:.2f} MB")
else:
    print("\n⚙️  Processing stdev features from scratch...")
    if use_cache and stdev_cache_file.exists():
        print("  (cache exists but force=True, reprocessing)")
    
    if wafer_list:
        stdev_pivot = process_parquet_batches(stdev_file, 'STD', wafer_list)
    else:
        df_stdev = pd.read_parquet(stdev_file)
        df_stdev['feature_name'] = df_stdev['SENSOR'] + "__" + df_stdev['STEP'] + "__STD"
        stdev_pivot = df_stdev.pivot_table(
            index='WAFER_SCRIBE',
            columns='feature_name',
            values='StDevValue',
            aggfunc='mean'
        )
        del df_stdev
        gc.collect()
    
    print(f"  Stdev features shape: {stdev_pivot.shape}")
    print(f"  Memory: {stdev_pivot.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    
    # Save to cache
    print(f"\n  💾 Saving to cache: {stdev_cache_file}")
    stdev_pivot.to_parquet(stdev_cache_file)
    print(f"  ✓ Cache saved ({stdev_cache_file.stat().st_size / 1e6:.2f} MB)")


⚙️  Processing stdev features from scratch...
  Reading fd_stddev.parquet...
  Processing 210 row groups...


 12%|█▏        | 26/210 [00:55<06:36,  2.15s/it]


  Converting to DataFrame...
  Stdev features shape: (16817, 1103)
  Memory: 148.73 MB

  💾 Saving to cache: outputs\cache\stdev_pivot.parquet
  ✓ Cache saved (37.88 MB)


In [9]:
# Combine
print("\nCombining mean and stdev features...")
sensor_features = pd.concat([mean_pivot, stdev_pivot], axis=1)
print(f"  Combined shape: {sensor_features.shape}")
del mean_pivot, stdev_pivot
gc.collect()


Combining mean and stdev features...
  Combined shape: (16817, 7637)


0

In [10]:
# Add missingness indicators
print("\nAdding missingness indicators...")
missing_cols = []
missing_data = []

for col in tqdm(sensor_features.columns, desc="Missingness"):
    missing_pct = sensor_features[col].isna().mean()
    if 0 < missing_pct < 0.5:
        missing_cols.append(col + "__MISSING")
        missing_data.append(sensor_features[col].isna().astype(np.int8))

if missing_cols:
    missing_df = pd.DataFrame(dict(zip(missing_cols, missing_data)), 
                             index=sensor_features.index)
    sensor_features = pd.concat([sensor_features, missing_df], axis=1)
    print(f"  Added {len(missing_cols)} missingness indicators")
    del missing_df
    gc.collect()


Adding missingness indicators...


Missingness: 100%|██████████| 7637/7637 [00:03<00:00, 2402.92it/s]


  Added 4434 missingness indicators


In [11]:
# Impute with per-column median
print("\nImputing missing values with per-column median...")
non_missing_cols = [c for c in sensor_features.columns if not c.endswith("__MISSING")]
for col in tqdm(non_missing_cols, desc="Imputing"):
    if sensor_features[col].isna().any():
        if sensor_features[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
            sensor_features[col] = sensor_features[col].fillna(
                sensor_features[col].median()
            )
        elif sensor_features[col].dtype == 'object' or sensor_features[col].dtype.name == 'string':
            mode_val = sensor_features[col].mode()[0] if len(sensor_features[col].mode()) > 0 else 'UNKNOWN'
            sensor_features[col] = sensor_features[col].fillna(mode_val)


Imputing missing values with per-column median...


Imputing: 100%|██████████| 7637/7637 [00:10<00:00, 699.68it/s]


In [12]:
# Drop near-zero variance columns
print("\nDropping near-zero variance columns (>95% identical)...")
initial_cols = len(sensor_features.columns)
cols_to_drop = []

sample_size = min(1000, len(sensor_features))
sample_idx = sensor_features.index[:sample_size]

for col in tqdm(sensor_features.columns, desc="Variance check"):
    if sensor_features[col].dtype in [np.float64, np.float32, np.float16, 
                                      np.int64, np.int32, np.int8]:
        nunique_sample = sensor_features.loc[sample_idx, col].nunique()
        if nunique_sample == 1:
            mode_freq = sensor_features[col].value_counts().iloc[0] if len(
                sensor_features[col].value_counts()) > 0 else 0
            if mode_freq / len(sensor_features) > 0.95:
                cols_to_drop.append(col)

sensor_features.drop(columns=cols_to_drop, inplace=True)
print(f"  Dropped {len(cols_to_drop)} columns ({initial_cols} -> {len(sensor_features.columns)})")


Dropping near-zero variance columns (>95% identical)...


Variance check: 100%|██████████| 12071/12071 [00:19<00:00, 613.97it/s]


  Dropped 2183 columns (12071 -> 9888)


## NEW: Create Column-to-Step Mapping for Horizon Models

This section creates `column_step_map.json` which maps every sensor feature column to its process SeqNo from `step_seq.csv`. This enables horizon-based filtering where models can use only features from steps up to a certain point in the process.

In [13]:
# Load step_seq.csv to map STEP to SeqNo
print("\n" + "=" * 60)
print("Creating column-to-step mapping for horizon models...")
print("=" * 60)

step_seq_file = data_dir / "step_seq.csv"
if not step_seq_file.exists():
    print(f"\nWARNING: {step_seq_file} not found.")
    print("Creating step_seq.csv from available data...")
    
    # Extract unique steps from sensor data and assign sequential numbers
    df_sample = pd.read_parquet(mean_file, columns=['STEP'])
    unique_steps = sorted(df_sample['STEP'].unique())
    
    step_seq_df = pd.DataFrame({
        'STEP': unique_steps,
        'SeqNo': range(1, len(unique_steps) + 1)
    })
    
    step_seq_df.to_csv(step_seq_file, index=False)
    print(f"  Created step_seq.csv with {len(unique_steps)} steps")
    del df_sample
    gc.collect()
else:
    step_seq_df = pd.read_csv(step_seq_file)
    print(f"  Loaded step_seq.csv with {len(step_seq_df)} steps")

# Create mapping from STEP to SeqNo
step_to_seqno = dict(zip(step_seq_df['STEP'], step_seq_df['SeqNo']))
print(f"  Step-to-SeqNo mapping: {len(step_to_seqno)} steps")


Creating column-to-step mapping for horizon models...

Creating step_seq.csv from available data...
  Created step_seq.csv with 240 steps
  Step-to-SeqNo mapping: 240 steps


In [14]:
# Create column_step_map for sensor features
column_step_map = {}

for col in sensor_features.columns:
    # Extract STEP from column name (format: SENSOR__STEP__SUFFIX)
    parts = col.split('__')
    
    if len(parts) >= 2:
        step = parts[1]
        
        # Look up SeqNo for this step
        if step in step_to_seqno:
            column_step_map[col] = int(step_to_seqno[step])
        else:
            # If step not found, assign 0 (always included)
            column_step_map[col] = 0
            print(f"  WARNING: Step '{step}' not found in step_seq.csv, assigning SeqNo=0")
    else:
        # If column doesn't follow expected format, assign 0
        column_step_map[col] = 0

print(f"\nCreated column_step_map for {len(column_step_map)} sensor features")
print(f"  SeqNo range: {min(column_step_map.values())} to {max(column_step_map.values())}")


Created column_step_map for 9888 sensor features
  SeqNo range: 1 to 240


In [15]:
# Save column_step_map.json
column_map_file = features_dir / "column_step_map.json"
with open(column_map_file, 'w') as f:
    json.dump(column_step_map, f, indent=2)

print(f"\n[OK] Saved column_step_map.json with {len(column_step_map)} entries")

# Show sample mappings
print("\nSample column-to-step mappings:")
for i, (col, seqno) in enumerate(list(column_step_map.items())[:5]):
    print(f"  {col}: SeqNo={seqno}")


[OK] Saved column_step_map.json with 9888 entries

Sample column-to-step mappings:
  APCPressure__WP0008__MEAN: SeqNo=147
  AirbagPressureArea4__CM0003__MEAN: SeqNo=3
  AsH3GasBottlePressure__IM0014__MEAN: SeqNo=69
  B5_BRANCH_VV_ActualTemp__DF0022__MEAN: SeqNo=49
  BiasMatchSeriesCapPosition__DE0005__MEAN: SeqNo=20


In [16]:
# Calculate sparsity
total_cells = sensor_features.shape[0] * sensor_features.shape[1]
missing_cells = sensor_features.isna().sum().sum()
sparsity = 100 * missing_cells / total_cells if total_cells > 0 else 0

# Save sensor features
output_file = features_dir / "sensor_features.parquet"
print(f"\nSaving sensor features to {output_file}...")
sensor_features.to_parquet(output_file)

# Print statistics
print("\n" + "=" * 60)
print("Sensor Feature Statistics:")
print("=" * 60)
print(f"Total wafers: {len(sensor_features)}")
print(f"Total features: {len(sensor_features.columns)}")
print(f"Sparsity: {sparsity:.2f}%")
print(f"Memory usage: {sensor_features.memory_usage(deep=True).sum() / 1e6:.2f} MB")

# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Sensor features complete in {elapsed:.2f}s")
print("=" * 60)


Saving sensor features to outputs\features\sensor_features.parquet...

Sensor Feature Statistics:
Total wafers: 16817
Total features: 9888
Sparsity: 0.02%
Memory usage: 987.18 MB

[OK] Sensor features complete in 1063.35s


---

## Cache Management

**Cache files saved:**
- `outputs/cache/mean_pivot.parquet` - Mean feature pivot table
- `outputs/cache/stdev_pivot.parquet` - Stdev feature pivot table

**To clear cache and reprocess:**
```python
force = True  # Set this in cell 3 and re-run
```

**To disable caching:**
```python
use_cache = False  # Set this in cell 3
```

**Cache benefits:**
- First run: Processes parquet files and saves cache (~20 minutes)
- Subsequent runs: Loads from cache (~10 seconds)
- Huge time savings when tweaking downstream steps!